***

# **Mortgage Lending Data**

***

In [ ]:
# packages

import pandas as pd
import os
from datetime import datetime

In [16]:
# let's make a function that will clean and shape our data the same way the Delaware guys did it
# https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?counties=06101,06115,06113,06061,06017,06067&years=2023

def get_demographics_new(row):
    demographics = []

    if row['derived_race'] in ['Black or African American', 'White', 'Asian']:
        demographics.append(row['derived_race'])
    else:
        demographics.append('Non-White')

    if row['derived_sex'] in ['Male', 'Female']:
        demographics.append(row['derived_sex'])

    if row['derived_ethnicity'] in ['Not Hispanic or Latino', 'Hispanic or Latino']:
        demographics.append(row['derived_ethnicity'])
    return demographics


def hmda_maker(years, counties):
    '''
    Years can only be from 2018, to 2024. County codes can be found by using the online tool: https://ffiec.cfpb.gov/data-browser/data/2023?category=counties. 
    '''

    dfs = []

    for year in years:
        if int(year) < 2018 or int(year) > int(datetime.now().year - 1): 
            print(f"Error: the year {year} does not have accessible data.") 
            continue

        url = 'https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?counties=' + ','.join(counties) + '&years=' + str(year)
        
        df = pd.read_csv(url)

        df = df[df['loan_purpose'].isin([1, 2, 31])]
        df = df[df['action_taken'].isin([1, 3])]
        df['loan_purpose'] = df['loan_purpose'].map({1: 'Home purchase', 2: 'Home improvement', 31: 'Refinancing'})
        df['demographics'] = df.apply(get_demographics_new, axis=1)

        df_exploud = df.explode('demographics')

        df_exploud['originations'] = df_exploud['action_taken'].apply(lambda x: 1 if x == 1 else 0)
        df_exploud['denials'] = df_exploud['action_taken'].apply(lambda x: 1 if x == 3 else 0)

        df_initial = df_exploud.groupby(['activity_year', 'county_code', 'demographics', 'loan_purpose']).agg(
            originated=('originations', 'sum'),
            denied=('denials', 'sum')
        ).reset_index()

        df_sums = df_initial.groupby(['activity_year', 'county_code', 'demographics']).agg(
            total_originated=('originated', 'sum'),
            total_denied=('denied', 'sum')
        ).reset_index()

        df_sums['loan_purpose'] = 'All'

        df_final = pd.concat([df_initial, df_sums[['activity_year', 'county_code', 'loan_purpose', 'demographics', 'total_originated', 'total_denied']]], ignore_index=True)
        df_final['originations'] = df_final['originated'].fillna(df_final['total_originated'])
        df_final['denials'] = df_final['denied'].fillna(df_final['total_denied'])
        df_final.drop(columns=['total_originated', 'total_denied'], inplace=True)
        df_final['county_name'] = df_final['county_code'].map({6101: 'Sutter', 6115: 'Yuba', 6113: 'Yolo', 6061: 'Placer', 6017: 'El Dorado', 6067: 'Sacramento'})

        dfs.append(df_final)
    
    # df_county = pd.concat(dfs, ignore_index=True)
    df_county['total'] = df_county['originations'] + df_county['denials']
    df_county['denial_rate'] = df_county['denials'] / df_county['total']
    df_county['origination_rate'] = df_county['originations'] / df_county['total']
    
    df_county.rename(columns = {'activity_year': 'year', 'county_code': 'county_id','loan_purpose': 'purpose', 'demographics': 'demographic'},inplace = True)

    df_county = df_county[['year', 'county_id', 'county_name', 'purpose', 'demographic', 'denials', 'originations', 'total', 'denial_rate', 'origination_rate']]
    return df_county

dates = ['2018', '2019', '2020', '2021', '2022', '2023']
codes = ['06101','06115','06113','06061','06017','06067']
df_new = hmda_maker(years= dates, counties=codes)

C:\Users\jchoy\AppData\Local\Temp\ipykernel_14240\3856134405.py:36: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url)
C:\Users\jchoy\AppData\Local\Temp\ipykernel_14240\3856134405.py:36: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url)
C:\Users\jchoy\AppData\Local\Temp\ipykernel_14240\3856134405.py:36: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url)
C:\Users\jchoy\AppData\Local\Temp\ipykernel_14240\3856134405.py:36: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url)
C:\Users\jchoy\AppData\Local\Temp\ipykernel_14240\3856134405.py:36: Dtyp

In [41]:
# Need to do this for the old data too

import os

def get_demographics_old(row):
    demographics = []
    
    if row['applicant_race_name_1'] in ['Black or African American', 'White', 'Asian']:
        demographics.append(row['applicant_race_name_1'])
    else:
        demographics.append('Non-White')
    # Add gender
    if row['applicant_sex_name'] in ['Male', 'Female']:
        demographics.append(row['applicant_sex_name'])
    # Latino 
    if row['applicant_ethnicity_name'] in ['Not Hispanic or Latino', 'Hispanic or Latino']:
        demographics.append(row['applicant_ethnicity_name'])
    return demographics

def vintage_data(path, counties):
    dfs = []

    for file_name in os.listdir(path):
        if file_name.endswith(".csv"):
            file_path = os.path.join(path, file_name)
            
            df = pd.read_csv(file_path)

            df = df[df['county_name'].isin(counties)]
            df = df[df['loan_purpose_name'].isin(['Home purchase', 'Home improvement', 'Refinancing'])]
            df = df[df['action_taken'].isin([1, 3])]
            df['loan_purpose'] = df['loan_purpose_name']
            df['demographics'] = df.apply(get_demographics_old, axis=1)

            df_exploud = df.explode('demographics')
            df_exploud['originations'] = df_exploud['action_taken'].apply(lambda x: 1 if x == 1 else 0)
            df_exploud['denials'] = df_exploud['action_taken'].apply(lambda x: 1 if x == 3 else 0)

            df_initial = df_exploud.groupby(['as_of_year', 'county_name', 'demographics', 'loan_purpose']).agg(
                originated=('originations', 'sum'),
                denied=('denials', 'sum')
            ).reset_index()

            df_sums = df_initial.groupby(['as_of_year', 'county_name', 'demographics']).agg(
                total_originated=('originated', 'sum'),
                total_denied=('denied', 'sum')
            ).reset_index()
            df_sums['loan_purpose'] = 'All'

            df_final = pd.concat([df_initial, df_sums[['as_of_year', 'county_name', 'loan_purpose', 'demographics', 'total_originated', 'total_denied']]], ignore_index=True)
            df_final['originations'] = df_final['originated'].fillna(df_final['total_originated'])
            df_final['denials'] = df_final['denied'].fillna(df_final['total_denied'])
            df_final.drop(columns=['total_originated', 'total_denied'], inplace=True)
            df_final['county_name'] = df_final['county_name'].map({'Sutter County': 'Sutter', 'Yuba County': 'Yuba', 'Yolo County': 'Yolo', 'Placer County': 'Placer', 'El Dorado County': 'El Dorado', 'Sacramento County': 'Sacramento'})
            df_final['county_id'] = df_final['county_name'].map({'Sutter': 6101, 'Yuba': 6115, 'Yolo': 6113, 'Placer': 6061, 'El Dorado': 6017, 'Sacramento': 6067})
            dfs.append(df_final)
        
    df_county = pd.concat(dfs, ignore_index=True)
    df_county['total'] = df_county['originations'] + df_county['denials']
    df_county['denial_rate'] = df_county['denials'] / df_county['total']
    df_county['origination_rate'] = df_county['originations'] / df_county['total']
    
    df_county.rename(columns = {'as_of_year': 'year', 'loan_purpose': 'purpose', 'demographics': 'demographic'}, inplace = True)

    df_county = df_county[['year', 'county_id', 'county_name', 'purpose', 'demographic', 'denials', 'originations', 'total', 'denial_rate', 'origination_rate']]
    return df_county


locs = ['Sacramento County', 'Placer County', 'El Dorado County', 'Yuba County', 'Yolo County', 'Sutter County']
path_data = "C://Users//jchoy//Documents//Python Projects//Regional-Monitoring//Indicator_Gen//Python Code//Mortgage Lending//Pre 2018 Data"
df_vintage = vintage_data(path=path_data, counties=locs)

C:\Users\jchoy\AppData\Local\Temp\ipykernel_14240\2386605871.py:27: DtypeWarning: Columns (34,36,38,44,46,48,57,59,61) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
C:\Users\jchoy\AppData\Local\Temp\ipykernel_14240\2386605871.py:27: DtypeWarning: Columns (34,36,38,42,44,46,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
C:\Users\jchoy\AppData\Local\Temp\ipykernel_14240\2386605871.py:27: DtypeWarning: Columns (34,36,38,44,46,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
C:\Users\jchoy\AppData\Local\Temp\ipykernel_14240\2386605871.py:27: DtypeWarning: Columns (34,36,38,42,44,46,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
C:\Users\jchoy\AppData\Local\Temp\ipykernel_14240\2386605871.py:27: DtypeWarning: Columns (34,36,38,44,46,48) have mixed types. S

In [ ]:
df_mortgage = pd.concat([df_vintage, df_new])
df_mortgage

# exports
# df_mortgage.to_csv('Mortgage_Lending_1.csv', index=False)
# df_mortgage.to_excel("Mortgage_Lending_1.xlsx")

,year,county_id,county_name,purpose,demographic,denials,originations,total,denial_rate,origination_rate
0,2007,6017,El Dorado,Home improvement,Asian,18.0,24.0,42.0,0.428571,0.571429
1,2007,6017,El Dorado,Home purchase,Asian,70.0,124.0,194.0,0.360825,0.639175
2,2007,6017,El Dorado,Refinancing,Asian,89.0,159.0,248.0,0.358871,0.641129
3,2007,6017,El Dorado,Home improvement,Black or African American,10.0,11.0,21.0,0.476190,0.523810
4,2007,6017,El Dorado,Home purchase,Black or African American,21.0,15.0,36.0,0.583333,0.416667
...,...,...,...,...,...,...,...,...,...,...
1146,2023,6115,Yuba,All,Hispanic or Latino,87.0,237.0,324.0,0.268519,0.731481
1147,2023,6115,Yuba,All,Male,111.0,397.0,508.0,0.218504,0.781496
1148,2023,6115,Yuba,All,Non-White,117.0,395.0,512.0,0.228516,0.771484
1149,2023,6115,Yuba,All,Not Hispanic or Latino,165.0,693.0,858.0,0.192308,0.807692


In [ ]:

# def demos(row):
#     # if row['derived_sex'] in ['Female', 'Male']:
#     #     return row['derived_sex']  # Prioritize gender if Male or Female
#     if row['derived_ethnicity'] == 'Hispanic or Latino':
#         return 'Latino'
#     elif row['derived_ethnicity'] == 'Not Hispanic or Latino':
#         return 'Non-Latino'
#     elif row['derived_race'] in ['Black or African American', 'White', 'Asian']:
#         return row['derived_race']
#     else:
#         return 'Non-White'  

    
# df_filtered = df[df['loan_purpose'].isin([1, 2, 31])]
# df_filtered = df[df['action_taken'].isin([1, 3])]
# df_filtered['loan_purpose'] = df['loan_purpose'].map({1: 'Home purchase', 2: 'Home improvement', 31: 'Refinancing'})
# df_filtered['demographics'] = df_filtered.apply(demos, axis=1)


# df_exploded = df_filtered.explode('demographics')


# df_exploded['originated'] = df_exploded['action_taken'].apply(lambda x: 1 if x == 1 else 0)
# df_exploded['denied'] = df_exploded['action_taken'].apply(lambda x: 1 if x == 3 else 0)

# result = df_exploded.groupby(['activity_year', 'county_code', 'demographics', 'loan_purpose']).agg(
#     originated=('originated', 'sum'),
#     denied=('denied', 'sum')
# ).reset_index()

# aggregated = result.groupby(['activity_year', 'county_code', 'demographics']).agg(
#     total_originated=('originated', 'sum'),
#     total_denied=('denied', 'sum')
# ).reset_index()

# aggregated['loan_purpose'] = 'All'

# df_with_totals = pd.concat([result, aggregated[['activity_year', 'county_code', 'loan_purpose', 'demographics', 'total_originated', 'total_denied']]], ignore_index=True)
# df_with_totals['originated'] = df_with_totals['originated'].fillna(df_with_totals['total_originated'])
# df_with_totals['denied'] = df_with_totals['denied'].fillna(df_with_totals['total_denied'])
# df_with_totals.drop(columns=['total_originated', 'total_denied'], inplace=True)

# df_with_totals['county_name'] = df_with_totals['county_code'].map({6101: 'Sutter', 6115: 'Yuba', 6113: 'Yolo', 6061: 'Placer', 6017: 'El Dorado', 6067: 'Sacramento'})

C:\Users\jchoy\AppData\Local\Temp\ipykernel_91932\3907832444.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['loan_purpose'] = df['loan_purpose'].map({1: 'Home purchase', 2: 'Home improvement', 31: 'Refinancing'})
C:\Users\jchoy\AppData\Local\Temp\ipykernel_91932\3907832444.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['demographics'] = df_filtered.apply(demos, axis=1)
